# WEIGHT1_LONG — 같은 강화학습을 충분히 학습하는 추가 실험

이 노트북은 **Colab에서만** 실행합니다. 기존 WEIGHT1의 모드·관측·보상·학습 알고리즘을 유지하고, 학습량과 반복 학습 수를 늘립니다. 기존 예비 실험의 미통과 결과는 그대로 보존합니다.

1. `FutureTraffic_WEIGHT1_Long.zip`을 Google Drive의 `MyDrive/FutureTraffic/`에 올립니다.
2. 이전 Colab 결과 `MyDrive/FutureTraffic/explore_weight_v1`을 그대로 둡니다. ZIP에 기존 모델 가중치는 들어 있지 않으며, 이 폴더의 42에피소드 모델을 읽어 재개합니다.
3. 새 Colab 런타임에서 **런타임 → 모두 실행**을 누릅니다. 새 결과는 `explore_weight_long_v1`에 저장합니다.

학습 난수 3개를 사용하고 자원에 맞춰 최대 3개를 병렬 실행합니다. 자원 프로필과 주 평가 시점은 결과를 보기 전에 고정합니다. 사용 가능한 CPU가 3개 이상이면 약 30만 단계, 2개면 약 20만 단계를 주 평가로 삼고, 1개이면 용량 부족으로 시작하지 않습니다. 기본 상한은 **실행 경과 24시간·기존 실행을 포함한 합산 계산 96시간**입니다. 중간 개발 평가를 함께 남기며, 마지막 모델들을 P0·M4와 비교합니다. 과거 예비 문턱만으로 일찍 끝내지 않습니다.

학습은 별도 프로세스로 실행하고 다음 칸이 로그를 보여줍니다. 실행 중인 로그 칸을 중단하면 실행기·학습 작업·SUMO를 함께 정리합니다. Colab 런타임 자체가 종료되면 프로세스도 종료될 수 있습니다. 이 경우 같은 노트북·동일한 결과 폴더로 다시 실행하면 저장된 경계에서 재개합니다. 맥에서 기존 실험을 재개하지 않습니다.


In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")
DRIVE_ROOT = Path("/content/drive/MyDrive/FutureTraffic")
ARCHIVE = DRIVE_ROOT / "FutureTraffic_WEIGHT1_Long.zip"
SOURCE = DRIVE_ROOT / "explore_weight_v1"
WORKSPACE = Path("/content/FutureTraffic_WEIGHT1_Long")
RESULTS = DRIVE_ROOT / "explore_weight_long_v1"
WORK = Path("/content/weight_long_work")
assert ARCHIVE.is_file(), f"새 Long ZIP을 먼저 업로드하세요: {ARCHIVE}"
assert SOURCE.is_dir() and (SOURCE / "experiment.json").is_file(), f"이전 WEIGHT1 결과가 없습니다: {SOURCE}"
assert (SOURCE / "models/seed_42").is_dir(), "이전 seed42 모델 폴더가 없습니다. 원래 결과 폴더를 옮기거나 지우지 마세요."
print("읽기 전용 출발 결과:", SOURCE)
print("새 결과 및 재개 기록:", RESULTS)


In [ ]:
# ZIP 전체의 파일 목록·SHA-256을 확인한 뒤 검증된 전용 추출기를 사용합니다.
# 기존 디렉터리를 지우지 않고 ZIP에 포함된 파일만 교체합니다.
import hashlib
import json
import stat
import sys
import types
import zipfile
from pathlib import PurePosixPath

with zipfile.ZipFile(ARCHIVE) as archive:
    names = archive.namelist()
    assert len(names) == len(set(names)) and names.count("manifest.json") == 1, "ZIP 파일 목록 오류"
    package_manifest = json.loads(archive.read("manifest.json"))
    assert package_manifest.get("kind") == "WEIGHT1_PACKAGE_V1", "검증할 수 없는 ZIP 형식입니다"
    assert package_manifest.get("experiment") == "WEIGHT1_LONG", "WEIGHT1_LONG 전용 ZIP이 아닙니다"
    assert package_manifest.get("source_models_included") is False, "기존 모델은 Drive 원본에서 읽어야 합니다"
    expected = package_manifest["files"]
    assert expected and "manifest.json" not in expected and set(names) == set(expected) | {"manifest.json"}, "ZIP과 manifest 목록 불일치"
    for info in archive.infolist():
        name, path = info.filename, PurePosixPath(info.filename)
        assert path.parts and not path.is_absolute() and ".." not in path.parts and path.as_posix() == name, "잘못된 ZIP 경로"
        assert "\\" not in name and not info.is_dir() and not stat.S_ISLNK(info.external_attr >> 16), "잘못된 ZIP 파일 형식"
        if name != "manifest.json":
            assert hashlib.sha256(archive.read(name)).hexdigest() == expected[name], f"SHA-256 불일치: {name}"
    source_code = archive.read("src/explore_weight_package.py").decode("utf-8")

bootstrap = types.ModuleType("weight1_long_verified_package")
bootstrap.__file__ = str(WORKSPACE / "src/explore_weight_package.py")
exec(compile(source_code, bootstrap.__file__, "exec"), bootstrap.__dict__)
package_manifest = bootstrap.extract_verified(ARCHIVE, WORKSPACE)
print(f"검증·추출 완료: {len(package_manifest['files'])}개 파일 → {WORKSPACE}")


In [ ]:
import os
import subprocess
import importlib.metadata

os.environ["FOCUSED_RULESET"] = "pet1"
os.environ["FOCUSED_NORTH_EXT"] = "240"
EXPECTED = {"numpy": "2.1.3", "torch": "2.11.0+cpu", "stable-baselines3": "2.9.0",
            "sumo-rl": "1.4.5", "eclipse-sumo": "1.27.1"}
source_packages = json.loads((SOURCE / "experiment.json").read_text())["packages"]
assert all(source_packages.get(name) == version for name, version in EXPECTED.items()), "출발 실험의 패키지 버전이 이 Long 묶음과 다릅니다."

def installed_versions():
    versions = {}
    for name in EXPECTED:
        try:
            versions[name] = importlib.metadata.version(name)
        except importlib.metadata.PackageNotFoundError:
            versions[name] = None
    return versions

before = installed_versions()
if before == EXPECTED:
    print("출발 실험과 패키지 버전이 모두 같아 재설치를 건너뜁니다.")
else:
    print("출발 실험과 같은 패키지 버전을 준비합니다:", before)
    # SUMO-RL은 requirements의 지정 Git commit을 유지합니다.
    subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"], cwd=WORKSPACE, check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "numpy==2.1.3", "stable-baselines3==2.9.0", "eclipse-sumo==1.27.1"],
                   cwd=WORKSPACE, check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "torch==2.11.0+cpu", "--index-url", "https://download.pytorch.org/whl/cpu"],
                   cwd=WORKSPACE, check=True)
after = installed_versions()
assert after == EXPECTED, f"패키지 버전이 출발 실험과 다릅니다: {after}"
print("실행 패키지 검증 통과:", after)


In [ ]:
# 기존 WEIGHT1 필수 검사와 새 Long 검사. 실제 학습은 아직 시작하지 않습니다.
subprocess.run([sys.executable, "-m", "unittest", "discover", "-s", "tests", "-p", "test_explore_weight*.py", "-v"],
               cwd=WORKSPACE, check=True)
COMMON = ["--source", str(SOURCE), "--out", str(RESULTS), "--work", str(WORK),
          "--workers", "auto", "--wall-hours", "24", "--compute-hours", "96"]
# 출발 모델·설정과 자원 프로필을 검증·동결합니다. 원래 WEIGHT1의 prepare/run을 호출하지 않습니다.
# 이미 같은 출력 폴더에서 실행 중이면 준비 명령을 중복 호출하지 않습니다.
record = RESULTS / "colab_process.json"
already_running = False
if record.is_file():
    try:
        pid = int(json.loads(record.read_text())["pid"])
        cmd = Path(f"/proc/{pid}/cmdline").read_bytes().replace(b"\0", b" ").decode()
        status = Path(f"/proc/{pid}/status").read_text()
        already_running = "explore_weight_long.py" in cmd and str(RESULTS) in cmd and "State:\tZ" not in status
    except (OSError, ValueError, KeyError):
        pass
if already_running:
    print("같은 Long 작업이 이미 실행 중입니다. 기존 로그 모니터로 연결합니다.")
else:
    subprocess.run([sys.executable, "src/explore_weight_long.py", "prepare", *COMMON], cwd=WORKSPACE, check=True)
    print("출발 결과와 Long 실행 설정 준비 완료")


In [ ]:
# 한 번 실행하면 단계별 학습·평가를 자동 진행합니다. 같은 노트북을 재개해도 중복 실행하지 않습니다.
import datetime
sys.path.insert(0, str(WORKSPACE / "src"))
from explore_weight_long_package import stop_owned_process_tree

RESULTS.mkdir(parents=True, exist_ok=True)
LOG = RESULTS / "colab_stdout.log"
PROCESS_RECORD = RESULTS / "colab_process.json"

def owned_long_pid():
    if not PROCESS_RECORD.is_file():
        return None
    try:
        pid = int(json.loads(PROCESS_RECORD.read_text())["pid"])
        command = Path(f"/proc/{pid}/cmdline").read_bytes().replace(b"\0", b" ").decode()
        status = Path(f"/proc/{pid}/status").read_text()
        if "explore_weight_long.py" in command and str(RESULTS) in command and "State:\tZ" not in status:
            return pid
    except (OSError, ValueError, KeyError):
        pass
    return None

existing_pid = owned_long_pid()
LONG_PROCESS = None
if existing_pid is not None:
    print("이미 실행 중인 Long 작업의 로그를 계속 봅니다. PID:", existing_pid)
else:
    try:
        with LOG.open("ab", buffering=0) as log:
            LONG_PROCESS = subprocess.Popen([sys.executable, "-u", "src/explore_weight_long.py", "run", *COMMON],
                                            cwd=WORKSPACE, stdout=log, stderr=subprocess.STDOUT,
                                            env={**os.environ, "PYTHONUNBUFFERED": "1"}, start_new_session=True)
        PROCESS_RECORD.write_text(json.dumps({"pid": LONG_PROCESS.pid,
            "started_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(), "log": str(LOG)}, indent=2))
    except BaseException:
        # 실행 직후 기록 저장에 실패하거나 이 칸을 중단해도 새로 만든 작업을 남기지 않습니다.
        if LONG_PROCESS is not None and LONG_PROCESS.poll() is None:
            stop_owned_process_tree(LONG_PROCESS.pid, RESULTS, graceful_seconds=30, terminate_seconds=5)
        raise
    print("Long 자동 실행을 시작했습니다. PID:", LONG_PROCESS.pid)
print("실시간 기록:", LOG)


In [ ]:
# 실행 중에는 30초마다 최근 로그를 갱신합니다. 이 칸이 끝나면 아래 요약 칸으로 넘어갑니다.
# 이 칸을 중단하면 실행기·학습 작업·SUMO를 함께 안전하게 멈춥니다.
import time
from IPython.display import clear_output
sys.path.insert(0, str(WORKSPACE / "src"))
from explore_weight_long_package import stop_owned_process_tree

try:
    while True:
        returncode = LONG_PROCESS.poll() if LONG_PROCESS is not None else None
        active_pid = owned_long_pid()
        if LOG.is_file():
            with LOG.open("rb") as stream:
                stream.seek(max(0, LOG.stat().st_size - 12000))
                recent = stream.read().decode("utf-8", errors="replace")
        else:
            recent = "아직 로그가 없습니다."
        clear_output(wait=True)
        print(recent)
        if active_pid is None or (LONG_PROCESS is not None and returncode is not None):
            print("실행 프로세스가 종료됐습니다. 종료 코드:", returncode)
            break
        print("실행 중 — 다음 갱신까지 30초. 결과:", RESULTS)
        time.sleep(30)
except KeyboardInterrupt:
    active_pid = owned_long_pid()
    if active_pid is not None:
        print("실행기에 중단을 알리고 저장·정산과 작업 종료를 기다립니다.")
        try:
            stopped = stop_owned_process_tree(active_pid, RESULTS, graceful_seconds=30, terminate_seconds=5)
        except KeyboardInterrupt:
            # 종료 대기 중 다시 중단한 경우에도 소유한 작업 그룹만 즉시 정리합니다.
            stopped = stop_owned_process_tree(active_pid, RESULTS, graceful_seconds=0, terminate_seconds=0)
        if LONG_PROCESS is not None:
            try:
                LONG_PROCESS.wait(timeout=3)
            except subprocess.TimeoutExpired:
                pass
        print("실행기·학습 작업·SUMO 중단:", stopped)
    else:
        print("실행 중인 Long 작업이 없습니다.")


In [ ]:
from IPython.display import Markdown, display

subprocess.run([sys.executable, "src/explore_weight_long.py", "summary", *COMMON], cwd=WORKSPACE, check=True)
summary_file = RESULTS / "summary.md"
if summary_file.is_file():
    display(Markdown(summary_file.read_text()))
else:
    print("요약 파일이 아직 없습니다. 실행 기록을 확인하세요:", LOG)
print("모델·원시 평가·예산 장부:", RESULTS)
